# C02u — Conservative lexical substitution revision
Fresh kernel. Reuses the **original C02s hyphen-repaired texts**, never C02t replacements. Separate checkpoints and results. No generation.

Changes: WordNet context-overlap sense selection; synonyms only (no arbitrary document vocabulary); inflected proposals only when a matching form occurs in the current document; exact POS/tag and count checks. A simple definition-overlap heuristic is **not reliable word-sense disambiguation**: accepted edits still require review. Ambiguous zero-overlap/tied senses are skipped.

Each edited word is scored using its own token path, conditioned on the preceding context, without averaging unchanged right-context words into its score. The .95 floor now applies to each word-path ratio and is a **new, uncalibrated conservative threshold**, not numerically comparable with C02t mutability. For pairs, both edits must pass in isolation and again in the combined text. The existing contextual-span score is an additional gate only.

Single proposals are scored in position-round-robin order under an explicit budget, then ranked by local likelihood. Pairs are assembled only from individually admissible substitutions. Bounded search can still miss candidates. Output PNG/PDF/CSV and manual reviews follow C02t. Requires the same environment and WordNet corpus.

In [ ]:
# Export every figure to reproduction/figures as well as the run directory.
from pathlib import Path as _FigurePath
def _export_figure(figure, filename, *args, **kwargs):
    original = _FigurePath(filename)
    original.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(original, *args, **kwargs)
    figure_dir = ROOT / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    try:
        suffix = original.relative_to(OUT).with_suffix("").as_posix().replace("/", "_")
    except ValueError:
        suffix = original.stem
    target = figure_dir / ("C02u_" + suffix + original.suffix)
    figure.savefig(target, *args, **kwargs)

from pathlib import Path
import json, hashlib, os, math, re, time, pickle
from collections import Counter
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy, torch, textstat, nltk
from nltk.corpus import wordnet as wn
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
assert (ROOT/'results').exists(), 'Open from repository root or experiments/'
BASE=ROOT/'results/causal/C02d2_fz118_ghostbuster_raid'
PRIOR=ROOT/'results/causal/C02e2_raid_feature_guided'
PARENT=ROOT/'results/causal/C02r_discovery_gated_punctuation'
PREV=ROOT/'results/causal/C02s_relative_likelihood_sensitivity'
INPUT=PREV/'wps_repair_v1/downstream_v1/hyphen_repair_v1/texts.csv'
OUT=ROOT/'results/causal/C02u_long_word_share_v2'
OUT.mkdir(parents=True,exist_ok=True)
CFG=dict(seed=20260908,long_min=7,max_steps=12,per_position=12,
         pair_pool_per_sign=24,max_scored=24,max_local_scored=128,mutability_floor=.95,
         cps_drift=3.,relative_punct_drift=.02)
# .95 is a NEW conservative mean-token contextual-score floor; it is not
# the .05 joint word-path probability floor used during generation.
wn.ensure_loaded()
nlp=spacy.load('en_core_web_sm')
normalize=lambda t:str(t).replace('\n',' ').replace('\r',' ')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def atomic(p,obj):
    p=Path(p);tmp=p.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));os.replace(tmp,p)
parent=json.loads((PARENT/'treatment_contract.json').read_text())
refs={r['case_id']:r for r in parent['prepared']}
texts=pd.read_csv(INPUT,keep_default_na=False)
assert texts.case_id.is_unique and len(texts)==27
selection=pd.read_csv(PARENT/'selection.csv')
assert set(texts.case_id)<=set(selection.loc[selection.partition.eq('discovery'),'case_id'])
corpus=pd.read_parquet(BASE/'paired_corpus.parquet')
features=pd.read_parquet(BASE/'fz118_paired_features.parquet')
text_map=corpus.set_index('source_id').text.to_dict()
train=features.loc[(features.dataset=='raid')&(features.split=='train')&(features.label==0)]
train_ids=set(train.source_id)
notebook=ROOT/'experiments/C02u_long_word_share.ipynb'
contract=dict(config=CFG,inputs={str(p.relative_to(ROOT)):sha(p) for p in
 [INPUT,PARENT/'treatment_contract.json',BASE/'paired_corpus.parquet',BASE/'fz118_paired_features.parquet',
 BASE/'fz118_feature_contract.json',PRIOR/'fz118_raid_model_features.json',PRIOR/'fz118_raid_no_parentheses_detector.pkl']},
 code=hashlib.sha256(json.dumps([c['source'] for c in json.loads(notebook.read_text())['cells'] if c['cell_type']=='code']).encode()).hexdigest(),
 spacy=nlp.meta,torch=torch.__version__,wordnet=wn.get_version(),scorer=parent['scorer'],revision=parent['scorer_revision'])
cp=OUT/'contract.json'
if cp.exists():assert json.loads(cp.read_text())==contract,'Contract changed: choose a new OUT'
else:atomic(cp,contract)
RUN_HASH=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
print('Reusing',len(texts),'texts from',INPUT)


In [ ]:
import re, pickle
from collections import Counter
import textstat
PUNCTUATION = [('comma', ','), ('period', '.'), ('colon', ':'), ('semicolon', ';'),
    ('question', '?'), ('exclamation', '!'), ('hyphen', '-'), ('quote', '"'),
    ('lparen', '('), ('rparen', ')'), ('lbracket', '['), ('rbracket', ']'), ('newline', '\n')]
google_words=[w.strip().lower() for w in (BASE/'google-10000-english.txt').read_text().splitlines() if w.strip()]
GOOGLE_TOP={n:set(google_words[:n]) for n in [100,1000,10000]}

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0


def clean_word_tokens(span):
    return [
        token.text.lower()
        for token in span
        if token.is_alpha
    ]


def clean_pos_tokens(span):
    return [
        token.pos_
        for token in span
        if token.is_alpha
    ]


def ngram_counter(sequence, n):
    if len(sequence) < n:
        return Counter()
    return Counter(
        tuple(sequence[i:i+n])
        for i in range(len(sequence) - n + 1)
    )


def multiset_overlap(left, right, n):
    a = ngram_counter(left, n)
    b = ngram_counter(right, n)
    denominator = min(sum(a.values()), sum(b.values()))
    if denominator == 0:
        return 0.0
    shared = sum((a & b).values())
    return float(shared / denominator)


def overlap_histogram(sentences, token_fn, n):
    if len(sentences) < 2:
        return np.zeros(10, dtype=np.float64)

    values = [
        multiset_overlap(token_fn(left), token_fn(right), n)
        for left, right in zip(sentences[:-1], sentences[1:])
    ]

    # Include 1.0 in the final bin.
    bins = np.minimum((np.asarray(values) * 10).astype(int), 9)
    counts = np.bincount(bins, minlength=10).astype(np.float64)
    return counts / max(1, len(values))


def conjunction_overlap(sentences, n):
    values = []
    for sentence in sentences:
        words = clean_word_tokens(sentence)
        for i, word in enumerate(words):
            if word == "and":
                values.append(
                    multiset_overlap(words[:i], words[i+1:], n)
                )
    return float(np.mean(values)) if values else 0.0


def punctuation_count(text, symbol_name, symbol):
    if symbol_name == "quote":
        return text.count('"') + text.count("“") + text.count("”")
    return text.count(symbol)


def finite_readability(fn, text):
    try:
        value = float(fn(text))
        return value if np.isfinite(value) else 0.0
    except Exception:
        return 0.0


def extract_fz118(text):
    text = str(text)
    doc = nlp(text)
    sentences = [sent for sent in doc.sents if clean_word_tokens(sent)]
    words = [token.text.lower() for token in doc if token.is_alpha]

    n_characters = len(text)
    n_words = len(words)
    n_sentences = len(sentences)
    n_syllables = int(textstat.syllable_count(text))
    n_difficult = int(textstat.difficult_words(text))
    n_short = sum(len(word) <= 3 for word in words)
    n_long = sum(len(word) >= 7 for word in words)

    values = [
        n_characters,
        n_syllables,
        n_words,
        n_sentences,
        n_difficult,
        n_short,
        n_long,
        safe_div(n_characters, n_words),
        safe_div(n_syllables, n_words),
        safe_div(n_words, n_sentences),
        safe_div(n_difficult, n_words),
        safe_div(n_short, n_words),
        safe_div(n_long, n_words),
        finite_readability(textstat.automated_readability_index, text),
        finite_readability(textstat.coleman_liau_index, text),
        finite_readability(textstat.flesch_kincaid_grade, text),
        finite_readability(textstat.flesch_reading_ease, text),
        finite_readability(textstat.gunning_fog, text),
        finite_readability(textstat.lix, text),
        finite_readability(textstat.mcalpine_eflaw, text),
        finite_readability(textstat.rix, text),
        finite_readability(textstat.smog_index, text),
        safe_div(sum(token.is_stop for token in doc if token.is_alpha), n_words),
        safe_div(len(set(words)), n_words),
        safe_div(sum(word in GOOGLE_TOP[100] for word in words), n_words),
        safe_div(sum(word in GOOGLE_TOP[1000] for word in words), n_words),
        safe_div(sum(word in GOOGLE_TOP[10000] for word in words), n_words),
    ]

    punct_counts = [
        punctuation_count(text, name, symbol)
        for name, symbol in PUNCTUATION
    ]
    values += [safe_div(count, n_characters) for count in punct_counts]
    values += [safe_div(count, n_sentences) for count in punct_counts]

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    paragraph_word_counts = [
        sum(token.is_alpha for token in nlp.make_doc(paragraph))
        for paragraph in paragraphs
    ]
    values += [
        len(paragraphs),
        float(np.mean(paragraph_word_counts)) if paragraph_word_counts else 0.0,
    ]

    for token_fn in [clean_word_tokens, clean_pos_tokens]:
        for n in [1, 2, 3]:
            values += overlap_histogram(sentences, token_fn, n).tolist()

    values += [conjunction_overlap(sentences, n) for n in [1, 2, 3]]

    vector = np.asarray(values, dtype=np.float64)
    assert vector.shape == (118,)
    assert np.isfinite(vector).all()
    return vector
# Exact tree-path contributions, summed across the frozen forest.
FEATURE_NAMES=json.loads((BASE/'fz118_feature_contract.json').read_text())['feature_names']
model_features=json.loads((PRIOR/'fz118_raid_model_features.json').read_text())
with (PRIOR/'fz118_raid_no_parentheses_detector.pkl').open('rb') as f:detector=pickle.load(f)
assert list(detector.classes_)==[0,1]
def contributions(X):
    a=np.asarray(X,dtype=np.float32); output=np.zeros_like(a,dtype=float);bias=0.
    for estimator in detector.estimators_:
        tree=estimator.tree_; value=tree.value[:,0,:].astype(float)
        prob=value[:,1]/value.sum(axis=1);bias+=prob[0]/len(detector.estimators_)
        for i,row in enumerate(a):
            node=0
            while tree.children_left[node]!=tree.children_right[node]:
                feature=tree.feature[node]
                child=tree.children_left[node] if row[feature]<=tree.threshold[node] else tree.children_right[node]
                output[i,feature]+=(prob[child]-prob[node])/len(detector.estimators_)
                node=child
    assert np.allclose(bias+output.sum(axis=1),detector.predict_proba(X)[:,1],atol=1e-8)
    return output


@lru_cache(maxsize=8192)
def vector_for(text): return extract_fz118(normalize(text))
def frame_for(texts):
    return pd.DataFrame([vector_for(t) for t in texts],columns=FEATURE_NAMES)


In [ ]:
class CachedNLP:
    def __init__(self,pipeline):
        self.pipeline=pipeline;self.parse=lru_cache(maxsize=128)(pipeline)
    def __call__(self,text):return self.parse(text)
    def __getattr__(self,name):return getattr(self.pipeline,name)
nlp=CachedNLP(nlp)
@lru_cache(maxsize=128)
def words(text):return [t for t in nlp(text) if t.is_alpha]
@lru_cache(maxsize=4096)
def profile(text):
    d=nlp(text);w=[t for t in d if t.is_alpha];ss=[s for s in d.sents if any(t.is_alpha for t in s)]
    return dict(W=len(w),V=len({t.text.lower() for t in w}),L=sum(len(t.text)>=7 for t in w),
                S=len(ss),C=len(text),sentence_words=[sum(t.is_alpha for t in s) for s in ss],
                cps=float(np.mean([len(s.text.strip()) for s in ss])))
def distance(x,band):return max(band[0]-x,0.,x-band[1])/max(band[1]-band[0],1e-12)
bands={};audit=[]
for r in texts.itertuples():
    ids=refs[r.case_id]['reference_ids'];assert len(ids)>=15 and set(ids)<=train_ids
    assert all(train.set_index('source_id').loc[sid,'domain']==r.domain for sid in ids)
    h=[profile(normalize(text_map[sid])) for sid in ids]
    bands[r.case_id]=np.quantile([x['L']/x['W'] for x in h],[.2,.6]).tolist()
    a=profile(r.text);lo,hi=bands[r.case_id]
    audit.append(dict(case_id=r.case_id,domain=r.domain,share=a['L']/a['W'],low=lo,high=hi,
                      direction='increase' if a['L']/a['W']<lo else 'decrease' if a['L']/a['W']>hi else 'in_band'))
pd.DataFrame(audit).to_csv(OUT/'initial_targets.csv',index=False)
atomic(OUT/'reference_ids.json',{r.case_id:refs[r.case_id]['reference_ids'] for r in texts.itertuples()})
display(pd.DataFrame(audit))


In [ ]:
SCORER_ID=parent['scorer'];SCORER_REVISION=parent['scorer_revision']
assert SCORER_REVISION and SCORER_REVISION!='main'
LEFT_WORDS=48;RIGHT_WORDS=32
tokenizer=AutoTokenizer.from_pretrained(SCORER_ID,revision=SCORER_REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(SCORER_ID,revision=SCORER_REVISION)
device='cuda' if torch.cuda.is_available() else 'cpu';model.to(device).eval()
# Score at most the model context; reject overflow rather than silently truncating.
@torch.inference_mode()
def continuation_score(left,tail):
    enc=tokenizer(left+tail,return_offsets_mapping=True,return_tensors='pt',add_special_tokens=False)
    offsets=enc.pop('offset_mapping')[0].tolist();ids=enc['input_ids'].to(device)
    if ids.shape[1]<2 or ids.shape[1]>model.config.max_position_embeddings:return None
    logp=model(input_ids=ids).logits[0,:-1].log_softmax(-1)
    chosen=logp.gather(1,ids[0,1:,None]).squeeze(1)
    mask=torch.tensor([end>len(left) for start,end in offsets[1:]],device=device)
    return float(chosen[mask].mean()) if mask.any() else None

def score_edit(text,e):
    a,b,replacement=e['start'],e['end'],e['replacement']
    left_parts=list(re.finditer(r'\S+',text[:a]));right_parts=list(re.finditer(r'\S+',text[b:]))
    left=text[left_parts[-LEFT_WORDS].start() if len(left_parts)>=LEFT_WORDS else 0:a]
    right=text[b:b+(right_parts[RIGHT_WORDS-1].end() if len(right_parts)>=RIGHT_WORDS else len(text)-b)]
    # Fixed flanks shared by both alternatives; future text contributes to each score.
    before=continuation_score(left,text[a:b]+right);after=continuation_score(left,replacement+right)
    if before is None or after is None:return None
    return dict(mutability=math.exp(min(0.,after-before)),score_before=before,score_after=after)



In [ ]:
def apply(text,ops):
    for a,b,w in sorted(ops,reverse=True):text=text[:a]+w+text[b:]
    return text

@lru_cache(maxsize=4096)
def sense_candidates(lemma,pos):
    return wn.synsets(lemma,pos=pos)

def proposals(text):
    ts=words(text);counts=Counter(t.text.lower() for t in ts);out=[]
    posmap={'NOUN':wn.NOUN,'VERB':wn.VERB,'ADJ':wn.ADJ,'ADV':wn.ADV}
    for t in ts:
        if t.pos_ not in posmap or t.ent_type_ or t.is_stop:continue
        senses=sense_candidates(t.lemma_,posmap[t.pos_])
        context={x.lemma_.lower() for x in t.sent if x.is_alpha and not x.is_stop and x.i!=t.i}
        ranked=[]
        for syn in senses:
            gloss=nlp(syn.definition()+' '+' '.join(syn.examples()))
            terms={x.lemma_.lower() for x in gloss if x.is_alpha and not x.is_stop}
            ranked.append((len(context & terms),syn))
        ranked.sort(key=lambda z:-z[0])
        if not ranked or ranked[0][0]==0:continue
        if len(ranked)>1 and ranked[0][0]==ranked[1][0]:continue
        lemmas={l.name().lower() for l in ranked[0][1].lemmas()}
        pool=set()
        for lemma in lemmas:
            if not lemma.isascii() or not lemma.isalpha():continue
            # Bare lemma only when source itself is uninflected.
            if t.text.lower()==t.lemma_.lower():pool.add(lemma)
            pool.update(u.text.lower() for u in ts if u.lemma_.lower()==lemma and u.tag_==t.tag_ and u.pos_==t.pos_)
        for w in sorted(pool):
            if w==t.text.lower():continue
            v=w.upper() if t.text.isupper() else w.capitalize() if t.text.istitle() else w
            out.append(dict(op=(t.idx,t.idx+len(t.text),v),dv=int(w not in counts)-int(counts[t.text.lower()]==1),
                            dl=int(len(w)>=7)-int(len(t.text)>=7),sense=ranked[0][1].name(),
                            group=('new' if w not in counts else 'seen')+'_'+('long' if len(w)>=7 else 'short')))
    return out

def valid(original,current,candidate,ops):
    a,b=profile(original),profile(candidate)
    if (a['W'],a['V'],a['sentence_words'])!=(b['W'],b['V'],b['sentence_words']):return 'word_unique_or_sentence_changed'
    if re.sub(r'[A-Za-z]+','',candidate)!=re.sub(r'[A-Za-z]+','',current):return 'nonword_changed'
    before=words(current);after=words(candidate)
    for x,y in zip(before,after):
        if x.text!=y.text and (x.pos_,x.tag_)!=(y.pos_,y.tag_):return 'pos_tag_changed'
    if abs(b['cps']-a['cps'])>CFG['cps_drift']+1e-12:return 'cps_drift'
    # Explicit drift bound relative to initial final text, not claimed parent-target attainment.
    if abs(a['C']/b['C']-1)>CFG['relative_punct_drift']+1e-12:return 'relative_punctuation_drift'
    return None

def score_ops(text,ops):
    a=min(x[0] for x in ops);b=max(x[1] for x in ops)
    replacement=apply(text[a:b],[(x-a,y-a,w) for x,y,w in ops])
    return score_edit(text,dict(start=a,end=b,replacement=replacement))

@torch.inference_mode()
def path_score(left,word):
    enc=tokenizer(left+word,return_offsets_mapping=True,return_tensors='pt',add_special_tokens=False)
    offsets=enc.pop('offset_mapping')[0].tolist();ids=enc['input_ids'].to(device)
    if ids.shape[1]<2 or ids.shape[1]>model.config.max_position_embeddings:return None
    # Reject unscorable first token; include any BPE token crossing the left boundary.
    if offsets[0][1]>len(left):return None
    mask=torch.tensor([end>len(left) for start,end in offsets[1:]],device=device)
    if not mask.any():return None
    lp=model(input_ids=ids).logits[0,:-1].log_softmax(-1).gather(1,ids[0,1:,None]).squeeze(1)
    return float(lp[mask].sum())

def local_ratio(text,op):
    a,b,w=op;parts=list(re.finditer(r'\S+',text[:a]))
    left=text[parts[-48].start() if len(parts)>=48 else 0:a]
    old=path_score(left,text[a:b]);new=path_score(left,w)
    if old is None or new is None:return None
    return math.exp(min(0.,new-old))

def run_case(cid,original):
    text=original;band=bands[cid];logs=[];edits=[];seen={text};status='step_budget'
    def loss(t):p=profile(t);return distance(p['L']/p['W'],band)
    for step in range(CFG['max_steps']):
        before_loss=loss(text)
        if before_loss<=1e-10:status='in_band';break
        raw=proposals(text);bypos={}
        for p in raw:bypos.setdefault(p['op'][0],[]).append(p)
        queue=[]
        while bypos:
            for pos in sorted(list(bypos)):
                queue.append(bypos[pos].pop(0))
                if not bypos[pos]:del bypos[pos]
        survivors=[]
        for j,p in enumerate(queue):
            rec=dict(step=step,ops=[p['op']],group=p['group'],sense=p['sense'],gain=0.,reason='local_scoring_budget');logs.append(rec)
            if j>=CFG['max_local_scored']:continue
            cand=apply(text,[p['op']]);bt=words(text);at=words(cand)
            if len(bt)!=len(at) or any((x.pos_,x.tag_)!=(y.pos_,y.tag_) for x,y in zip(bt,at) if x.text!=y.text):
                rec['reason']='pos_tag_changed';continue
            ratio=local_ratio(text,p['op']);rec['local_ratio']=ratio
            if ratio is None:rec['reason']='score_unavailable';continue
            if ratio<CFG['mutability_floor']:rec['reason']='local_likelihood_floor';continue
            rec['reason']='local_pass';survivors.append({**p,'ratio':ratio})
            if j%16==0:print(cid,'step',step,'local candidates',j+1,'/',min(len(queue),CFG['max_local_scored']),flush=True)
        candidates=[([p['op']],p['group']) for p in survivors if p['dv']==0]
        order=lambda p:(-p['ratio'],p['op'])
        plus=sorted([p for p in survivors if p['dv']==1],key=order)[:CFG['pair_pool_per_sign']]
        minus=sorted([p for p in survivors if p['dv']==-1],key=order)[:CFG['pair_pool_per_sign']]
        for a in plus:
            for b in minus:
                if a['op'][0]!=b['op'][0]:candidates.append(([a['op'],b['op']],a['group']+' + '+b['group']))
        eligible=[]
        for ops,group in candidates:
            cand=apply(text,ops);reason='visited' if cand in seen else valid(original,text,cand,ops);gain=before_loss-loss(cand)
            if not reason and gain<=1e-10:reason='no_target_improvement'
            rec=dict(step=step,ops=ops,group=group,gain=gain,reason=reason);logs.append(rec)
            if not reason:
                quality=min(p['ratio'] for p in survivors if p['op'] in ops)
                eligible.append((quality,gain,ops,cand,rec))
        eligible.sort(key=lambda z:(-z[0],-z[1],len(z[2]),str(z[2])))
        accepted=[]
        for j,(_,gain,ops,cand,rec) in enumerate(eligible):
            if j>=CFG['max_scored']:rec['reason']='scoring_budget';continue
            ratios=[]
            # Re-evaluate each substitution with the other edit already present.
            for op in ops:
                others=[x for x in ops if x!=op];context=apply(text,others)
                shift=sum(len(w)-(b-a) for a,b,w in others if a<op[0])
                ratios.append(local_ratio(context,(op[0]+shift,op[1]+shift,op[2])))
            if any(r is None or r<CFG['mutability_floor'] for r in ratios):rec['reason']='combined_local_floor';continue
            score=score_ops(text,ops)
            if score is None:rec['reason']='score_unavailable';continue
            rec.update(score);rec['local_ratios']=ratios
            if score['mutability']<CFG['mutability_floor']:rec['reason']='contextual_floor';continue
            rec['reason']='eligible';accepted.append((min(ratios),gain,ops,cand,rec))
        if not accepted:status='no_eligible_in_bounded_search';break
        _,_,ops,cand,rec=max(accepted,key=lambda z:(z[0],z[1],-len(z[2])))
        rec['reason']='accepted';edits.append(dict(before=text,after=cand,**rec));text=cand;seen.add(text)
        print(cid,'accepted step',step+1,'remaining distance',loss(text),flush=True)
    if loss(text)<=1e-10:status='in_band'
    return dict(case_id=cid,text=text,status=status,edits=edits,diagnostics=logs,loss=loss(text))


## Run the revised search
At most 128 single-word local comparisons and 24 combined contextual comparisons per step. Checkpoints resume under the frozen v2 contract. Same input-based character drift bounds as v1. Quality review remains pending.

In [ ]:
results=[]
for row in texts.itertuples():
    path=OUT/f'{row.case_id}.json'
    if path.exists():
        saved=json.loads(path.read_text());assert saved['run_hash']==RUN_HASH
        result=saved['result']
    else:
        print('Starting',row.case_id,flush=True)
        result=run_case(row.case_id,row.text)
        atomic(path,dict(run_hash=RUN_HASH,result=result))
    results.append(result)
    print(row.case_id,result['status'],'edits',len(result['edits']),flush=True)
final=texts.copy();final['text']=[r['text'] for r in results]
final.to_csv(OUT/'texts.csv',index=False)
review=[dict(case_id=r['case_id'],**e,fidelity_review='',grammar_review='',notes='') for r in results for e in r['edits']]
if not (OUT/'manual_review.csv').exists():pd.DataFrame(review,columns=['case_id','before','after','ops','group','mutability','fidelity_review','grammar_review','notes']).to_csv(OUT/'manual_review.csv',index=False)
pd.DataFrame([dict(case_id=r['case_id'],**d) for r in results for d in r['diagnostics']]).to_csv(OUT/'candidate_diagnostics.csv',index=False)
rows=[]
for old,new in zip(texts.itertuples(),final.itertuples()):
    a,b=profile(old.text),profile(new.text)
    assert a['W']==b['W'] and a['V']==b['V'] and a['sentence_words']==b['sentence_words']
    rows.append(dict(case_id=old.case_id,before=a['L']/a['W'],after=b['L']/b['W'],
                     cps_change=b['cps']-a['cps'],relative_punctuation_change=a['C']/b['C']-1,
                     unique_change=b['V']/b['W']-a['V']/a['W']))
audit=pd.DataFrame(rows);audit.to_csv(OUT/'attainment_and_drift.csv',index=False);display(audit)
fig,ax=plt.subplots(figsize=(9,4));ax.plot(audit.case_id,audit.before,'o',label='Before');ax.plot(audit.case_id,audit.after,'x',label='After')
ax.tick_params(axis='x',rotation=90);ax.set_ylabel('Long-word share');ax.legend();fig.tight_layout()
for ext in ('png','pdf'):_export_figure(fig,OUT/f'long_word_share.{ext}',dpi=300,bbox_inches='tight')
plt.show()


## Frozen detector evaluation
Same saved human cohort and weights as C02s. Human scores are used only here, never in proposal acceptance. ROC/TPR are descriptive discovery diagnostics; uncertainty in the human population is not estimated. Contributions are exact tree-path attributions, not causal feature effects.

In [ ]:
from sklearn.metrics import roc_curve,auc
human=pd.read_csv(PREV/'roc_discovery/human_scores_and_weights.csv')
assert human.source_id.is_unique
assert set(human.source_id).isdisjoint(train_ids)
HX=features.set_index('source_id').loc[human.source_id,model_features]
h=detector.predict_proba(HX)[:,1];hw=human.weight.to_numpy(dtype=float,copy=True);hw/=hw.sum()
np.testing.assert_allclose(h,human.AI_score.to_numpy(),atol=1e-8)
arrays={};Cs={};tables=[]
for label,df in [('Before',texts),('Long-word operation',final)]:
    X=frame_for(df.text.tolist())[model_features];arrays[label]=detector.predict_proba(X)[:,1];Cs[label]=contributions(X)
    tables.append(X.assign(case_id=df.case_id.to_numpy(),condition=label))
pd.concat(tables).to_csv(OUT/'features.csv',index=False)
pd.DataFrame(arrays,index=pd.Index(texts.case_id,name='case_id')).to_csv(OUT/'AI_scores.csv')
cuts=np.r_[np.unique(h)[::-1],-np.inf];fprs=np.array([hw[h>t].sum() for t in cuts])
budgets=np.array([.01,.05,.10,.20,.30,.50,1.]);idx=[np.flatnonzero(fprs<=b+1e-12)[-1] for b in budgets]
op=pd.DataFrame(dict(requested_FPR=budgets,achieved_FPR=fprs[idx],threshold=cuts[idx]))
fig,axes=plt.subplots(1,2,figsize=(11,4));rocrows=[];aucs=[]
for label,a in arrays.items():
    f,t,threshold=roc_curve(np.r_[np.zeros(len(h)),np.ones(len(a))],np.r_[h,a],sample_weight=np.r_[hw,np.ones(len(a))/len(a)],drop_intermediate=False)
    aucs.append(dict(condition=label,AUC=auc(f,t)))
    rocrows.extend(dict(condition=label,FPR=x,TPR=y,threshold=z) for x,y,z in zip(f,t,threshold))
    op[label]=[(a>cuts[j]).mean() for j in idx]
    for ax in axes:ax.step(f,t,where='post',label=f'{label} ({auc(f,t):.3f})')
for ax in axes:ax.plot([0,1],[0,1],'--',color='gray');ax.set(xlabel='Human FPR',ylabel='AI TPR');ax.legend()
axes[1].set_xlim(0,.2);fig.tight_layout()
for ext in ('png','pdf'):_export_figure(fig,OUT/f'ROC_staircase.{ext}',dpi=300,bbox_inches='tight')
pd.DataFrame(rocrows).to_csv(OUT/'ROC_points.csv',index=False);pd.DataFrame(aucs).to_csv(OUT/'ROC_AUC.csv',index=False);op.to_csv(OUT/'TPR_at_FPR.csv',index=False)
display(op);plt.show()
HC=pd.DataFrame(contributions(HX),columns=model_features).groupby(human.domain.to_numpy()).mean().loc[texts.domain].to_numpy()
gap=Cs['Long-word operation']-HC
rng=np.random.default_rng(CFG['seed']);draw=rng.integers(0,len(texts),(3000,len(texts)))
lo,hi=np.quantile(gap[draw].mean(1),[.025,.975],axis=0)
rank=pd.DataFrame(dict(feature=model_features,before=(Cs['Before']-HC).mean(0),after=gap.mean(0),ci_low=lo,ci_high=hi)).sort_values('after',ascending=False)
rank.to_csv(OUT/'contributions.csv',index=False)
top=rank.head(30).iloc[::-1];fig,ax=plt.subplots(figsize=(9,9));y=np.arange(len(top))
ax.hlines(y,top.ci_low,top.ci_high,color='pink');ax.scatter(top.before,y,label='Before');ax.scatter(top.after,y,label='After');ax.set_yticks(y,top.feature);ax.axvline(0,color='gray');ax.legend();ax.set_xlabel('Contribution minus fixed same-genre human reference');fig.tight_layout()
for ext in ('png','pdf'):_export_figure(fig,OUT/f'contributions.{ext}',dpi=300,bbox_inches='tight')
plt.show()
delta=arrays['Long-word operation']-arrays['Before'];ci=np.quantile(delta[draw].mean(1),[.025,.975])
pd.DataFrame([dict(mean_change=delta.mean(),ci_low=ci[0],ci_high=ci[1],cases=len(delta))]).to_csv(OUT/'paired_score_change.csv',index=False)
print('Saved:',OUT,'— quality review remains required.')
